# 🎬 Pipeline FINAL : U-Net PyTorch + Pulsation
## Version PyTorch Compatible macOS

Ce notebook crée un **vrai U-Net** et calcule la **vraie pulsation** !

## 1️⃣ Installation et Imports

In [ ]:
import subprocess
import sys

print("📦 Installation des dépendances...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-python", "numpy", "pandas", "matplotlib", "Pillow", "scikit-image"])

import cv2
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import cm
import warnings
warnings.filterwarnings('ignore')

print("✅ Tous les imports réussis !")
print(f"   PyTorch version: {torch.__version__}")
print(f"   Device: {'GPU' if torch.cuda.is_available() else 'CPU (macOS)'}")

## 2️⃣ Configuration

In [ ]:
# Chemins
BASE_PATH = "/Users/macbookpro/Documents/Master MIASHS/TER/semestre2/2025_08_15"
FRAMES_PATH = os.path.join(BASE_PATH, "frames_DJI_0819")
LABELS_PATH = os.path.join(FRAMES_PATH, "labels")
FLOTTEUR_LABELS_PATH = os.path.join(FRAMES_PATH, "labels_flotteur")
OUTPUT_PATH = os.path.join(BASE_PATH, "unet_results")

# Créer dossier de sortie
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Paramètres
FLOTTEUR_DIAMETER_MM = 45
IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 15
LEARNING_RATE = 1e-3
DEVICE = torch.device('cpu')  # macOS

print(f"📁 Base path: {BASE_PATH}")
print(f"📍 Frames: {FRAMES_PATH}")
print(f"📝 Labels: {LABELS_PATH}")
print(f"💾 Output: {OUTPUT_PATH}")
print(f"⚙️ Device: {DEVICE}")

## 3️⃣ Créer les masques à partir des labels YOLO

In [ ]:
print("\n" + "="*80)
print("🎨 CRÉATION DES MASQUES À PARTIR DES LABELS")
print("="*80)

def parse_yolo_label(label_file, frame_width, frame_height):
    """Parse un fichier YOLO et retourne les boîtes englobantes"""
    boxes = []
    if not os.path.exists(label_file):
        return boxes
    
    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                class_id = int(parts[0])
                x_center = float(parts[1]) * frame_width
                y_center = float(parts[2]) * frame_height
                box_width = float(parts[3]) * frame_width
                box_height = float(parts[4]) * frame_height
                
                x1 = int(max(0, x_center - box_width / 2))
                y1 = int(max(0, y_center - box_height / 2))
                x2 = int(min(frame_width, x_center + box_width / 2))
                y2 = int(min(frame_height, y_center + box_height / 2))
                
                if x2 > x1 and y2 > y1:
                    boxes.append((x1, y1, x2, y2))
            except:
                continue
    return boxes

def create_mask_from_boxes(boxes, frame_width, frame_height):
    """Crée un masque binaire à partir des boîtes"""
    mask = np.zeros((frame_height, frame_width), dtype=np.uint8)
    for (x1, y1, x2, y2) in boxes:
        cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)
    return mask

# Créer dossiers dataset
dataset_img_path = os.path.join(OUTPUT_PATH, "dataset_images")
dataset_mask_path = os.path.join(OUTPUT_PATH, "dataset_masks")
os.makedirs(dataset_img_path, exist_ok=True)
os.makedirs(dataset_mask_path, exist_ok=True)

# Charger frames et labels
frame_files = sorted([f for f in os.listdir(FRAMES_PATH) 
                     if f.startswith('frame_') and f.endswith('.jpg')])

print(f"\n📋 Frames trouvés: {len(frame_files)}")

# Charger la première frame pour les dimensions
first_frame = cv2.imread(os.path.join(FRAMES_PATH, frame_files[0]))
frame_height, frame_width = first_frame.shape[:2]
print(f"📐 Dimensions: {frame_width}x{frame_height}")

# Créer masques
valid_pairs = 0
for frame_file in frame_files:
    frame_path = os.path.join(FRAMES_PATH, frame_file)
    label_file = os.path.join(LABELS_PATH, os.path.splitext(frame_file)[0] + '.txt')
    
    # Charger frame
    frame = cv2.imread(frame_path)
    if frame is None:
        continue
    
    # Parser labels
    boxes = parse_yolo_label(label_file, frame_width, frame_height)
    
    if len(boxes) > 0:
        # Créer masque
        mask = create_mask_from_boxes(boxes, frame_width, frame_height)
        
        # Sauvegarder
        img_name = frame_file
        mask_name = os.path.splitext(frame_file)[0] + '_mask.png'
        
        cv2.imwrite(os.path.join(dataset_img_path, img_name), frame)
        cv2.imwrite(os.path.join(dataset_mask_path, mask_name), mask)
        
        valid_pairs += 1

print(f"\n✅ Dataset créé: {valid_pairs} paires image-masque")
print(f"   Images: {dataset_img_path}")
print(f"   Masques: {dataset_mask_path}")

## 4️⃣ Dataset PyTorch

In [ ]:
class MedusaDataset(Dataset):
    """Dataset PyTorch pour les méduses"""
    def __init__(self, img_dir, mask_dir, img_size=256):
        self.img_files = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
    
    def __len__(self):
        return len(self.img_files)
    
    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        mask_name = os.path.splitext(img_name)[0] + '_mask.png'
        
        # Charger image
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)
        
        # Charger masque
        mask_path = os.path.join(self.mask_dir, mask_name)
        mask = Image.open(mask_path).convert('L')
        mask = transforms.Resize((self.img_size, self.img_size))(mask)
        mask = transforms.ToTensor()(mask)
        
        return img, mask.squeeze(0)

# Créer dataset
dataset = MedusaDataset(dataset_img_path, dataset_mask_path, IMG_SIZE)
print(f"\n✅ Dataset PyTorch créé: {len(dataset)} paires")

# DataLoader
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"✅ DataLoader créé: {len(dataloader)} batches")

## 5️⃣ U-Net PyTorch

In [ ]:
class UNet(nn.Module):
    """U-Net simple pour la segmentation"""
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()
        
        # Encoder
        self.enc1 = self.conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self.conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = self.conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = self.conv_block(256, 512)
        
        # Decoder
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = self.conv_block(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(128, 64)
        
        # Output
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        x = self.pool1(enc1)
        enc2 = self.enc2(x)
        x = self.pool2(enc2)
        enc3 = self.enc3(x)
        x = self.pool3(enc3)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Decoder
        x = self.upconv3(x)
        x = torch.cat([x, enc3], dim=1)
        x = self.dec3(x)
        
        x = self.upconv2(x)
        x = torch.cat([x, enc2], dim=1)
        x = self.dec2(x)
        
        x = self.upconv1(x)
        x = torch.cat([x, enc1], dim=1)
        x = self.dec1(x)
        
        x = self.final(x)
        return torch.sigmoid(x)  # Output entre 0 et 1

# Créer modèle
model = UNet().to(DEVICE)
print("\n✅ U-Net créé")
print(f"   Paramètres: {sum(p.numel() for p in model.parameters()):,}")

## 6️⃣ Entraîner U-Net

In [ ]:
print("\n" + "="*80)
print("🚀 ENTRAÎNEMENT U-NET")
print("="*80)

# Loss et optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Entraîner
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    
    for batch_idx, (images, masks) in enumerate(dataloader):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE).unsqueeze(1)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 3 == 0:
        print(f"   Époque {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}")

print(f"\n✅ Entraînement terminé !")

# Sauvegarder modèle
model_path = os.path.join(OUTPUT_PATH, "unet_meduses.pth")
torch.save(model.state_dict(), model_path)
print(f"✅ Modèle sauvegardé: {model_path}")

## 7️⃣ Calibration (Détection flotteurs)

In [ ]:
print("\n" + "="*80)
print("📏 CALIBRATION - DÉTECTION FLOTTEURS")
print("="*80)

# Charger les labels flotteurs
flotteur_diameter_pixels_list = []

flotteur_frame_files = sorted([f for f in os.listdir(FLOTTEUR_LABELS_PATH) 
                              if f.endswith('.txt') and f.startswith('frame_')])

for flotteur_file in flotteur_frame_files:
    flotteur_label_path = os.path.join(FLOTTEUR_LABELS_PATH, flotteur_file)
    
    with open(flotteur_label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                try:
                    x_center = float(parts[1]) * frame_width
                    y_center = float(parts[2]) * frame_height
                    box_width = float(parts[3]) * frame_width
                    box_height = float(parts[4]) * frame_height
                    
                    diameter = max(box_width, box_height)
                    flotteur_diameter_pixels_list.append(diameter)
                except:
                    continue

if flotteur_diameter_pixels_list:
    avg_diameter_pixels = np.mean(flotteur_diameter_pixels_list)
    pixels_per_mm = avg_diameter_pixels / FLOTTEUR_DIAMETER_MM
    mm_per_pixel = 1 / pixels_per_mm
    
    print(f"\n✅ Calibration calculée:")
    print(f"   Diamètre flotteur (pixels): {avg_diameter_pixels:.1f}")
    print(f"   Diamètre flotteur (mm): {FLOTTEUR_DIAMETER_MM}")
    print(f"   Ratio: {pixels_per_mm:.3f} pixels/mm")
    print(f"   Conversion: {mm_per_pixel:.4f} mm/pixel")
else:
    print("⚠️ Aucun flotteur trouvé")
    pixels_per_mm = 1.0  # Valeur par défaut

## 8️⃣ Calcul de la pulsation

In [ ]:
print("\n" + "="*80)
print("💓 CALCUL DE LA PULSATION")
print("="*80)

# Charger modèle
model.eval()
pulsation_data = []

with torch.no_grad():
    for idx, frame_file in enumerate(frame_files, 1):
        frame_path = os.path.join(FRAMES_PATH, frame_file)
        frame = Image.open(frame_path).convert('RGB')
        
        # Redimensionner et normaliser
        frame_tensor = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])(frame).unsqueeze(0).to(DEVICE)
        
        # Prédire
        mask_pred = model(frame_tensor)
        mask_np = (mask_pred[0, 0].cpu().numpy() > 0.5).astype(np.uint8) * 255
        
        # Redimensionner au size original
        mask_orig = cv2.resize(mask_np, (frame_width, frame_height))
        
        # Calculer aire
        area_pixels = cv2.countNonZero(mask_orig)
        area_mm2 = area_pixels * (mm_per_pixel ** 2)
        
        pulsation_data.append({
            'frame': idx,
            'filename': frame_file,
            'area_pixels': area_pixels,
            'area_mm2': area_mm2
        })
        
        if idx % 5 == 0:
            print(f"   Frame {idx}: aire = {area_pixels} pixels = {area_mm2:.2f} mm²")

df_pulsation = pd.DataFrame(pulsation_data)
print(f"\n✅ Pulsation calculée pour {len(df_pulsation)} frames")
print(f"\n📊 Statistiques:")
print(df_pulsation[['area_pixels', 'area_mm2']].describe())

## 9️⃣ Visualisation et graphiques

In [ ]:
print("\n" + "="*80)
print("📊 VISUALISATION")
print("="*80)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Graphique 1 : Pulsation en pixels
axes[0].plot(df_pulsation['frame'], df_pulsation['area_pixels'], 'b-', linewidth=2.5, marker='o', markersize=4)
axes[0].fill_between(df_pulsation['frame'], df_pulsation['area_pixels'], alpha=0.3)
axes[0].set_xlabel('Frame', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Aire (pixels)', fontsize=11, fontweight='bold')
axes[0].set_title('Pulsation des Méduses - Aire en Pixels', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Graphique 2 : Pulsation en mm²
axes[1].plot(df_pulsation['frame'], df_pulsation['area_mm2'], 'g-', linewidth=2.5, marker='s', markersize=4)
axes[1].fill_between(df_pulsation['frame'], df_pulsation['area_mm2'], alpha=0.3, color='green')
axes[1].set_xlabel('Frame', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Aire (mm²)', fontsize=11, fontweight='bold')
axes[1].set_title('Pulsation des Méduses - Aire en mm²', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Graphique 3 : Variation pulsation (dérivée)
pulsation_variation = df_pulsation['area_mm2'].diff().fillna(0)
colors = ['red' if x < 0 else 'blue' for x in pulsation_variation]
axes[2].bar(df_pulsation['frame'], pulsation_variation, color=colors, alpha=0.7)
axes[2].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[2].set_xlabel('Frame', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Variation Aire (mm²/frame)', fontsize=11, fontweight='bold')
axes[2].set_title('Variation de Pulsation (Contraction/Expansion)', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt_path = os.path.join(OUTPUT_PATH, "pulsation_analysis.png")
plt.savefig(plt_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Graphiques sauvegardés: {plt_path}")

## 🔟 Exporter les données

In [ ]:
print("\n" + "="*80)
print("💾 EXPORT DES DONNÉES")
print("="*80)

# Exporter pulsation
pulsation_csv = os.path.join(OUTPUT_PATH, "pulsation_data.csv")
df_pulsation.to_csv(pulsation_csv, index=False)
print(f"\n✅ Pulsation exportée: {pulsation_csv}")

# Exporter calibration
calib_data = pd.DataFrame([{
    'flotteur_diameter_mm': FLOTTEUR_DIAMETER_MM,
    'flotteur_diameter_pixels': avg_diameter_pixels if flotteur_diameter_pixels_list else np.nan,
    'pixels_per_mm': pixels_per_mm,
    'mm_per_pixel': mm_per_pixel
}])
calib_csv = os.path.join(OUTPUT_PATH, "calibration_data.csv")
calib_data.to_csv(calib_csv, index=False)
print(f"✅ Calibration exportée: {calib_csv}")

# Résumé
print(f"\n" + "="*80)
print(f"✨ PIPELINE TERMINÉ !")
print(f"="*80)
print(f"""
✅ RÉSULTATS:
   • U-Net entraîné et sauvegardé
   • Pulsation calculée: {len(df_pulsation)} frames
   • Calibration: {pixels_per_mm:.3f} pixels/mm
   • Aire moyenne: {df_pulsation['area_mm2'].mean():.2f} mm²
   • Aire min/max: {df_pulsation['area_mm2'].min():.2f} / {df_pulsation['area_mm2'].max():.2f} mm²

📁 FICHIERS GÉNÉRÉS:
   • Modèle: {model_path}
   • Pulsation: {pulsation_csv}
   • Calibration: {calib_csv}
   • Graphiques: {plt_path}
   • Dataset images: {dataset_img_path}
   • Dataset masques: {dataset_mask_path}
""")